In [1]:
!pip install matplotlib numpy ipywidgets

In [4]:
# Kód pro DDA algoritmus

def zaokrouhli(hodnota):
    """Klasické matematické zaokrouhlení používané v počítačové grafice (0.5 jde vždy nahoru)."""
    return int(hodnota + 0.5)

def dda_didactic(x0, y0, x1, y1):
    """
    DDA (Digital Differential Analyzer) algoritmus pro rasterizaci úsečky se záznamem kroků.
    Vrací seznam slovníků, kde každý slovník popisuje jeden krok rasterizace.
    """
    dx = x1 - x0
    dy = y1 - y0

    # Počet kroků podle delší osy
    kroky = max(abs(dx), abs(dy))

    # Přírůstky pro jeden krok (v desetinných číslech)
    x_inc = dx / kroky
    y_inc = dy / kroky

    x = float(x0)
    y = float(y0)
    
    steps = []
    pixels = []

    for k in range(kroky + 1):
        px = zaokrouhli(x)
        py = zaokrouhli(y)
        pixels.append((px, py))

        steps.append({
            "krok": k,
            "presne_xy": (round(x, 2), round(y, 2)),
            "aktualni_pixel": (px, py),
            "vykreslene_pixely": list(pixels),
            "y_inc": round(y_inc, 2),
            "dalsi_presne_y": round(y + y_inc, 2)
        })

        x += x_inc
        y += y_inc

    return steps

In [7]:
# Kód pro vykreslení algoritmu do grafu

import matplotlib.pyplot as plt
import matplotlib.patches as patches
import ipywidgets as widgets
from IPython.display import display

# Pevně dané souřadnice a předpočítání kroků
X0, Y0 = 1, 1
X1, Y1 = 11, 8
historie = dda_didactic(X0, Y0, X1, Y1)
max_krok = len(historie) - 1

aktualni_krok = 0

# Vytvoření ovládacích prvků
btn_zpet = widgets.Button(description='<- Krok zpět', disabled=True, button_style='info')
btn_vpred = widgets.Button(description='Krok vpřed ->', disabled=False, button_style='info')
lbl_stav = widgets.Label(value=f"Krok: 0 / {max_krok}")
vystup_grafu = widgets.Output(layout=widgets.Layout(min_height='620px'))

def vykresli_aktualni_stav():
    data_kroku = historie[aktualni_krok]
    cur_x, cur_y = data_kroku["aktualni_pixel"]
    presne_x, presne_y = data_kroku["presne_xy"]
    y_inc = data_kroku["y_inc"]
    dalsi_y = data_kroku["dalsi_presne_y"]

    max_x = X1 + 3
    max_y = Y1 + 3

    # Popis nad graf
    if aktualni_krok == max_krok:
        nadpis = (
            f"Krok {aktualni_krok} z {max_krok} | Jsme v cílovém pixelu [{cur_x}, {cur_y}]\n"
            f"Přesná výška Y = {presne_y:.2f} → po zaokrouhlení dostáváme Y = {cur_y}. Úsečka je kompletně vykreslená."
        )
    else:
        dalsi_pixel_y = zaokrouhli(dalsi_y)
        nadpis = (
            f"Krok {aktualni_krok} z {max_krok} | Stojíme na pixelu [{cur_x}, {cur_y}]\n"
            f"Přesná desetinná výška Y je {presne_y:.2f} → po zaokrouhlení na celé číslo dostáváme Y = {cur_y} (pixel [{cur_x}, {cur_y}]).\n"
            f"Pro další krok přičteme +{y_inc}: nové přesné Y = {presne_y:.2f} + {y_inc} = {dalsi_y:.2f} → zaokrouhlí se na Y = {dalsi_pixel_y} (pixel [{cur_x + 1}, {dalsi_pixel_y}])."
        )

    with vystup_grafu:
        vystup_grafu.clear_output(wait=True)

        fig, ax = plt.subplots(figsize=(9, 6))

        # Vykreslení již rasterizovaných pixelů
        for px, py in data_kroku["vykreslene_pixely"]:
            je_aktualni = (px, py) == (cur_x, cur_y)
            barva = '#ff6b6b' if je_aktualni else '#4ecdc4'
            alfa = 0.9 if je_aktualni else 0.5

            rect = patches.Rectangle((px - 0.5, py - 0.5), 1, 1, 
                                     linewidth=1.5, edgecolor='black', 
                                     facecolor=barva, alpha=alfa, zorder=2)
            ax.add_patch(rect)

        # Ideální matematická úsečka
        ax.plot([X0, X1], [Y0, Y1], color='#1a535c', linestyle='--', linewidth=2, label='Ideální úsečka', zorder=3)
        ax.scatter([X0, X1], [Y0, Y1], color='#1a535c', s=60, zorder=4)

        # Střed aktuálního pixelu (zaokrouhlená hodnota)
        ax.scatter([cur_x], [cur_y], color='red', s=40, zorder=5, label=f'Střed pixelu po zaokrouhlení [{cur_x}, {cur_y}]')

        # Přesná desetinná poloha vypočtená DDA algoritmem
        ax.scatter([presne_x], [presne_y], color='#ffd166', edgecolor='black', s=80, zorder=6, 
                   label=f'Přesná desetinná hodnota [{presne_x}, {presne_y:.2f}]')
        
        # Spojnice ukazující zaokrouhlení z desetinného bodu do středu pixelu
        ax.plot([presne_x, cur_x], [presne_y, cur_y], color='red', linestyle=':', linewidth=1.5, zorder=4)

        # Nastavení mřížky
        ax.set_xticks(range(0, max_x))
        ax.set_yticks(range(0, max_y))
        ax.set_xticks([x - 0.5 for x in range(0, max_x + 1)], minor=True)
        ax.set_yticks([y - 0.5 for y in range(0, max_y + 1)], minor=True)

        ax.grid(which='minor', color='#cccccc', linestyle='-', linewidth=1)
        ax.grid(which='major', color='#eeeeee', linestyle=':', linewidth=0.5)

        ax.set_xlim(-0.5, max_x - 0.5)
        ax.set_ylim(-0.5, max_y - 0.5)
        ax.set_aspect('equal')

        # Panel nad grafem
        ax.set_title(
            nadpis, 
            fontsize=10.5, pad=14, loc='left',
            bbox=dict(boxstyle='round,pad=0.6', facecolor='#f8f9fa', edgecolor='#dee2e6')
        )

        ax.legend(loc='upper left')
        plt.tight_layout()
        plt.show()

# Obsluha kliknutí na tlačítka
def pri_kliknuti_zpet(b):
    global aktualni_krok
    if aktualni_krok > 0:
        aktualni_krok -= 1
        aktualizuj_ui()

def pri_kliknuti_vpred(b):
    global aktualni_krok
    if aktualni_krok < max_krok:
        aktualni_krok += 1
        aktualizuj_ui()

def aktualizuj_ui():
    btn_zpet.disabled = (aktualni_krok == 0)
    btn_vpred.disabled = (aktualni_krok == max_krok)
    lbl_stav.value = f"Krok: {aktualni_krok} / {max_krok}"
    vykresli_aktualni_stav()

btn_zpet.on_click(pri_kliknuti_zpet)
btn_vpred.on_click(pri_kliknuti_vpred)

# Složení panelu a vykreslení
ovladaci_panel = widgets.HBox([btn_zpet, btn_vpred, lbl_stav])
#display(vystup_grafu, ovladaci_panel)
#vykresli_aktualni_stav()

## Princip a pseudokód algoritmu DDA

Algoritmus **DDA (Digital Differential Analyzer)** vykresluje úsečku přímým výpočtem z diferenciální rovnice přímky pomocí **desetinných čísel**. V každém kroku se posune o 1 pixel v hlavní ose $X$ a k souřadnici $Y$ přičte konstantní desetinný přírůstek. Výsledný pixel na mřížce získá **zaokrouhlením** desetinné hodnoty $Y$ na nejbližší celé číslo.

```text
ALGORITMUS: DDA algoritmus
VSTUP:  Počáteční bod [x0, y0], koncový bod [x1, y1]
VÝSTUP: Vykreslené pixely úsečky

1. Spočítej vzdálenosti v osách a počet kroků:
   dx = x1 - x0
   dy = y1 - y0
   kroky = max(|dx|, |dy|)

2. Vypočítej desetinný přírůstek pro jeden krok:
   x_inc = dx / kroky
   y_inc = dy / kroky

3. Nastav výchozí souřadnice:
   x = x0
   y = y0

4. Pro k od 0 do kroky opakuj:
   a) Vykresli_Pixel(zaokrouhli(x), zaokrouhli(y))
   
   b) Posuň se na další pozici:
      x = x + x_inc
      y = y + y_inc
```

In [8]:
display(vystup_grafu, ovladaci_panel)
vykresli_aktualni_stav()

Output(layout=Layout(min_height='620px'))